# YZTA 2026 Datathon — 

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

SEED     = 42
N_FOLDS  = 5   # 56K satır için 5-fold yeterli, daha hızlı
TARGET   = 'bilissel_performans_skoru'
BASE_PATH = '/kaggle/input/competitions/yzta-2026-datathon/'

print('✅ Hazır.')

✅ Hazır.


## 1. Veri Yükleme

In [2]:
train = pd.read_csv(BASE_PATH + 'train.csv')
test  = pd.read_csv(BASE_PATH + 'test_x.csv')
sample_sub = pd.read_csv(BASE_PATH + 'sample_submission.csv')

test_ids = test['id'].copy()

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'\nHedef istatistikler:')
print(train[TARGET].describe())

Train: (56000, 24), Test: (24000, 23)

Hedef istatistikler:
count    56000.000000
mean         5.913096
std          2.231759
min          0.000000
25%          4.397431
50%          6.032249
75%          7.574980
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


## 2. Ülke Adı Normalizasyonu (Senin veride "Spain" var, diğerleri Türkçe)

In [3]:
ulke_map = {
    'Spain': 'Ispanya', 'Germany': 'Almanya', 'France': 'Fransa',
    'Japan': 'Japonya', 'China': 'Cin', 'UK': 'Ingiltere',
    'United Kingdom': 'Ingiltere', 'USA': 'Amerika',
    'Australia': 'Avustralya', 'Italy': 'Italya',
    'Brazil': 'Brezilya', 'Canada': 'Kanada',
    'India': 'Hindistan', 'Russia': 'Rusya',
    'Mexico': 'Meksika', 'Portugal': 'Portekiz',
}
train['ulke'] = train['ulke'].replace(ulke_map)
test['ulke']  = test['ulke'].replace(ulke_map)
print('Ülkeler:', sorted(train['ulke'].dropna().unique()))

Ülkeler: ['Amerika', 'Arjantin', 'Cin', 'Fransa', 'Guney Kore', 'Ingiltere', 'Ispanya', 'Isvec', 'Meksika', 'Netherlands', 'Portekiz', 'South Korea', 'Sweden', 'Yeni Zelanda']


## 3. Feature Engineering (Senin 44 feature'ına ek)

In [4]:
def feature_engineering(df):
    df = df.copy()

    # Uyku kalitesi
    df['kaliteli_uyku']     = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['hafif_uyku']        = 100 - df['kaliteli_uyku']
    df['rem_derin_oran']    = df['rem_yuzdesi'] / (df['derin_uyku_yuzdesi'] + 1e-5)
    df['log_uyku_dalma']    = np.log1p(df['uykuya_dalma_suresi_dk'])
    df['zor_uyuyan']        = (df['uykuya_dalma_suresi_dk'] > 30).astype(int)
    df['hic_uyanmayan']     = (df['gecelik_uyanma_sayisi'] == 0).astype(int)
    df['cok_uyanan']        = (df['gecelik_uyanma_sayisi'] >= 3).astype(int)
    df['uyku_bozuklugu']    = df['uykuya_dalma_suresi_dk'] * (1 + df['gecelik_uyanma_sayisi'])
    df['log_uyku_bozuklugu']= np.log1p(df['uyku_bozuklugu'])

    # Stres
    df['stres_kare']        = df['stres_skoru'] ** 2
    df['dusuk_stres']       = (df['stres_skoru'] < 4).astype(int)
    df['yuksek_stres']      = (df['stres_skoru'] > 7).astype(int)

    # Fiziksel aktivite
    df['log_adim']          = np.log1p(df['gunluk_adim_sayisi'])
    df['aktif']             = (df['gunluk_adim_sayisi'] > 8000).astype(int)
    df['hareketsiz']        = (df['gunluk_adim_sayisi'] < 3000).astype(int)

    # Kafein & ekran
    df['log_kafein']        = np.log1p(df['uyku_oncesi_kafein_mg'])
    df['kafein_var']        = (df['uyku_oncesi_kafein_mg'] > 0).astype(int)
    df['log_ekran']         = np.log1p(df['uyku_oncesi_ekran_suresi_dk'])
    df['kafein_ekran']      = df['uyku_oncesi_kafein_mg'] + df['uyku_oncesi_ekran_suresi_dk'] * 0.5

    # BMI
    df['bmi_kat']           = pd.cut(df['vucut_kitle_indeksi'],
                                     bins=[0,18.5,25,30,100], labels=[0,1,2,3]).astype(float)
    df['normal_kilo']       = ((df['vucut_kitle_indeksi'] >= 18.5) &
                               (df['vucut_kitle_indeksi'] < 25)).astype(int)

    # Yaş
    df['yas_kare']          = df['yas'] ** 2
    df['yas_grubu']         = pd.cut(df['yas'], bins=[0,25,35,45,55,120],
                                     labels=[0,1,2,3,4]).astype(float)

    # Çalışma
    df['fazla_mesai']       = (df['gunluk_calisma_saati'] > 9).astype(int)
    df['az_calisma']        = (df['gunluk_calisma_saati'] < 4).astype(int)

    # Nabız
    df['dusuk_nabiz']       = (df['dinlenik_nabiz_bpm'] < 60).astype(int)
    df['yuksek_nabiz']      = (df['dinlenik_nabiz_bpm'] > 80).astype(int)

    # Oda sıcaklığı
    df['optimal_sicaklik']  = ((df['oda_sicakligi_celsius'] >= 18) &
                               (df['oda_sicakligi_celsius'] <= 20)).astype(int)
    df['sicaklik_sapma']    = (df['oda_sicakligi_celsius'] - 19).abs()

    # Hafta sonu
    df['uyku_farki_abs']    = df['hafta_sonu_uyku_farki_saat'].abs()
    df['sosyal_jet_lag']    = (df['hafta_sonu_uyku_farki_saat'].abs() > 1.5).astype(int)

    # Şekerleme
    df['sekerleme_var']     = (df['sekerleme_suresi_dk'] > 0).astype(int)
    df['log_sekerleme']     = np.log1p(df['sekerleme_suresi_dk'])

    # ── ÇAPRAZ ÖZELLİKLER ─────────────────────────────
    df['stres_x_uyku_boz']  = df['stres_skoru'] * df['uyku_bozuklugu']
    df['stres_x_kaliteli']  = df['stres_skoru'] * (100 - df['kaliteli_uyku'])
    df['adim_stres_oran']   = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1)
    df['yas_stres']         = df['yas'] * df['stres_skoru']
    df['kafein_x_dalma']    = df['uyku_oncesi_kafein_mg'] * df['uykuya_dalma_suresi_dk']

    return df

train = feature_engineering(train)
test  = feature_engineering(test)
print(f'FE sonrası: Train {train.shape}, Test {test.shape}')

FE sonrası: Train (56000, 62), Test (24000, 61)


## 4. KFold Target Encoding (Label Encoding'den çok daha güçlü)

In [5]:
def kfold_target_encode(train_df, test_df, cat_cols, target, n_splits=5, smooth=20):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    global_mean = train_df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cat_cols:
        new_col = f'{col}_te'
        train_df[new_col] = global_mean

        for tr_idx, val_idx in kf.split(train_df):
            tr = train_df.iloc[tr_idx]
            stats = tr.groupby(col)[target].agg(['mean', 'count'])
            stats['smoothed'] = (
                (stats['mean'] * stats['count'] + global_mean * smooth) /
                (stats['count'] + smooth)
            )
            train_df.loc[train_df.index[val_idx], new_col] = (
                train_df.iloc[val_idx][col].map(stats['smoothed']).fillna(global_mean)
            )

        stats_all = train_df.groupby(col)[target].agg(['mean', 'count'])
        stats_all['smoothed'] = (
            (stats_all['mean'] * stats_all['count'] + global_mean * smooth) /
            (stats_all['count'] + smooth)
        )
        test_df[new_col] = test_df[col].map(stats_all['smoothed']).fillna(global_mean)
        print(f'  ✅ {col} → {new_col}')

    return train_df, test_df

TE_COLS = ['meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'cinsiyet', 'mevsim', 'gun_tipi']
print('Target Encoding...')
train, test = kfold_target_encode(train, test, TE_COLS, TARGET)

Target Encoding...
  ✅ meslek → meslek_te
  ✅ ulke → ulke_te
  ✅ kronotip → kronotip_te
  ✅ ruh_sagligi_durumu → ruh_sagligi_durumu_te
  ✅ cinsiyet → cinsiyet_te
  ✅ mevsim → mevsim_te
  ✅ gun_tipi → gun_tipi_te


## 5. Preprocessing 

In [6]:
# Train + Test birleştir (senin yapın gibi) — ama TARGET hariç
train['_is_train'] = 1
test['_is_train']  = 0
test[TARGET]       = np.nan

df = pd.concat([train, test], axis=0).reset_index(drop=True)

# Eksik değer: sayısal → medyan, kategorik → mod
for col in df.columns:
    if col in ['id', '_is_train', TARGET]:
        continue
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Label Encoding — train+test birlikte fit (senin yaklaşımın, doğru)
cat_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Split
EXCLUDE = ['id', '_is_train', TARGET]
feature_cols = [c for c in df.columns if c not in EXCLUDE]

df_train = df[df['_is_train'] == 1].reset_index(drop=True)
df_test  = df[df['_is_train'] == 0].reset_index(drop=True)

X      = df_train[feature_cols].values
y      = df_train[TARGET].values
X_test = df_test[feature_cols].values

print(f'Feature sayısı: {len(feature_cols)}')
print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')

Feature sayısı: 67
X: (56000, 67), y: (56000,), X_test: (24000, 67)


## 6. Model Parametreleri 

In [7]:
lgb_params = {
    'objective'       : 'regression',
    'metric'          : 'rmse',
    'n_estimators'    : 5000,
    'learning_rate'   : 0.01,    # daha düşük lr → daha iyi genelleme
    'num_leaves'      : 127,
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'reg_alpha'       : 0.1,
    'reg_lambda'      : 0.1,
    'random_state'    : SEED,
    'n_jobs'          : -1,
    'verbose'         : -1
}

xgb_params = {
    'objective'           : 'reg:squarederror',
    'eval_metric'         : 'rmse',
    'n_estimators'        : 5000,
    'learning_rate'       : 0.01,
    'max_depth'           : 6,
    'subsample'           : 0.8,
    'colsample_bytree'    : 0.8,
    'min_child_weight'    : 5,
    'early_stopping_rounds': 100,  # ✅ düzeltildi
    'tree_method'         : 'hist',
    'random_state'        : SEED,
    'n_jobs'              : -1,
    'verbosity'           : 0
}

cat_params = {
    'loss_function'  : 'RMSE',
    'iterations'     : 5000,
    'learning_rate'  : 0.01,
    'depth'          : 7,
    'early_stopping_rounds': 100,
    'random_seed'    : SEED,
    'verbose'        : 0
}

print('✅ Parametreler hazır.')

✅ Parametreler hazır.


## 7. OOF Eğitim (5-Fold)

In [8]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

preds_lgb = np.zeros(len(X_test))
preds_xgb = np.zeros(len(X_test))
preds_cat = np.zeros(len(X_test))

lgb_fold_scores = []
xgb_fold_scores = []
cat_fold_scores = []

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f'\nFold {fold+1}/{N_FOLDS}')

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    # LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    preds_lgb += model_lgb.predict(X_test) / N_FOLDS
    s = rmse(y_val, oof_lgb[val_idx])
    lgb_fold_scores.append(s)
    print(f'  LGB: {s:.4f}')

    # XGBoost — early_stopping_rounds artık parametrede tanımlı
    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    preds_xgb += model_xgb.predict(X_test) / N_FOLDS
    s = rmse(y_val, oof_xgb[val_idx])
    xgb_fold_scores.append(s)
    print(f'  XGB: {s:.4f}')

    # CatBoost
    model_cat = cb.CatBoostRegressor(**cat_params)
    model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = model_cat.predict(X_val)
    preds_cat += model_cat.predict(X_test) / N_FOLDS
    s = rmse(y_val, oof_cat[val_idx])
    cat_fold_scores.append(s)
    print(f'  CAT: {s:.4f}')

print(f'\n✅ LGB OOF RMSE: {rmse(y, oof_lgb):.5f} | Ort: {np.mean(lgb_fold_scores):.4f}')
print(f'✅ XGB OOF RMSE: {rmse(y, oof_xgb):.5f} | Ort: {np.mean(xgb_fold_scores):.4f}')
print(f'✅ CAT OOF RMSE: {rmse(y, oof_cat):.5f} | Ort: {np.mean(cat_fold_scores):.4f}')


Fold 1/5
  LGB: 1.2314
  XGB: 1.2416
  CAT: 1.2239

Fold 2/5
  LGB: 1.2269
  XGB: 1.2310
  CAT: 1.2190

Fold 3/5
  LGB: 1.2129
  XGB: 1.2196
  CAT: 1.2046

Fold 4/5
  LGB: 1.2234
  XGB: 1.2329
  CAT: 1.2125

Fold 5/5
  LGB: 1.2463
  XGB: 1.2789
  CAT: 1.2361

✅ LGB OOF RMSE: 1.22825 | Ort: 1.2282
✅ XGB OOF RMSE: 1.24096 | Ort: 1.2408
✅ CAT OOF RMSE: 1.21927 | Ort: 1.2192


## 8. Stacking

In [9]:
oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_cat])
test_stack = np.column_stack([preds_lgb, preds_xgb, preds_cat])

# ✅ DOĞRU: cross_val_predict ile leak-free OOF
meta = Ridge(alpha=1.0)
stack_oof   = cross_val_predict(meta, oof_stack, y, cv=KFold(N_FOLDS, shuffle=True, random_state=SEED))
stack_rmse  = rmse(y, stack_oof)

# Test için tüm OOF ile fit
meta.fit(oof_stack, y)
final_preds = meta.predict(test_stack)
final_preds = np.clip(final_preds, 0, 10)

print(f'\n🔗 STACK OOF RMSE: {stack_rmse:.5f}')
print(f'   Meta katsayılar (lgb, xgb, cat): {meta.coef_.round(4)}')

# Ağırlıklı blend karşılaştırma
rmse_arr = np.array([rmse(y, oof_lgb), rmse(y, oof_xgb), rmse(y, oof_cat)])
w = (1 / rmse_arr) / (1 / rmse_arr).sum()
blend_oof   = oof_stack @ w
blend_preds = test_stack @ w
blend_rmse  = rmse(y, blend_oof)
print(f'⚖️  Blend OOF RMSE: {blend_rmse:.5f} | w={w.round(3)}')

# En iyi seçim
if blend_rmse < stack_rmse:
    final_preds = np.clip(blend_preds, 0, 10)
    best_method = f'Blend ({blend_rmse:.5f})'
else:
    best_method = f'Stack ({stack_rmse:.5f})'
print(f'\n🏆 Seçilen: {best_method}')


🔗 STACK OOF RMSE: 1.21890
   Meta katsayılar (lgb, xgb, cat): [ 0.2341 -0.1518  0.9221]
⚖️  Blend OOF RMSE: 1.22356 | w=[0.334 0.33  0.336]

🏆 Seçilen: Stack (1.21890)


## 9. Submission

In [10]:
TARGET = "bilissel_performans_skoru"

submission = pd.DataFrame({
    "id": test_ids,
    TARGET: np.clip(final_preds, 0, 10)
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("✅ Submission shape:", submission.shape)
print(submission.head())

✅ Submission shape: (24000, 2)
   id  bilissel_performans_skoru
0   1                   5.974269
1   2                   6.435404
2   3                   2.933747
3   4                   7.162415
4   5                   3.615162
